# Programmation Bash - exercices - éléments de correction

Les réponses proposées ci-dessous ne sont que des éléments de correction. D'autres solutions alternatives sont parfaitement possibles.

## Préparation des données

Supprimez tout d'abord les éventuels fichiers résiduels en exécutant la cellule suivante :

In [ ]:
#IN [1]
rm -rf genomes genomes.gz staphylo.tgz

Téléchargez le fichier `genomes.tgz` avec la commande :

In [ ]:
#IN [2]
wget https://raw.githubusercontent.com/yvesclement/unix/main/genomes.gz


*Remarque.* Si la commande `wget` n'est pas disponible, essayez d'utiliser la commande `curl` :
```
curl -o genomes.gz https://raw.githubusercontent.com/yvesclement/unix/main/genomes.gz
```

Ce fichier contient les fichiers GenBank tronqués de quelques organismes (les dix premières lignes seulement).

## Découverte du jeu de données

1. Décompressez l'archive `genomes.tgz` que vous avez téléchargée.

Utilisez pour cela la commande `tar`. Vous trouverez un rappel des différentes options [ici](https://wizardzines.com/comics/tar/).

Pour décompresser l'archive `genomes.tgz`, il faut faire deux opérations :

1. Décompresser l'archive.
2. Désarchiver les fichiers contenus dans l'archive.

Ces opérations peuvent se faire avec deux commandes :
```
gzip -d genomes.tgz    # ou gunzip genomes.tgz
tar xf genomes.tar
```

Mais ces deux opérations (décompression et désarchivage) peuvent aussi se faire en une seule commande :

In [ ]:
#IN [3]
tar -zxf genomes.gz

In [ ]:
#IN [4]
ls -alt

Pour un rappel de la signification des options `z`, `x` et `g` de la commande `tar`, consultez le site [explainshell](https://explainshell.com/explain?cmd=tar+zxf+genomes.tgz).

Un répertoire `genomes` est alors créé :

In [ ]:
#IN [5]
ls -lh | grep genomes

2. Combien de fichiers contient le répertoire `genomes` qui a été créé ? Utilisez, une (ou plusieurs) commande(s) Unix pour répondre. 😎

Pour déterminer le nombre de fichiers dans le répertoire `genomes`, on peut utiliser la commande `ls -l` qui affiche un fichier par ligne puis compter le nombre de lignes :

In [ ]:
#IN [6]
ls -l genomes/ | wc -l

Mais il faut faire attention car la commande `ls -l` affiche effectivement un fichier par ligne mais aussi la taille total des fichiers (ici `total 376`) au début :

In [ ]:
#IN [7]
ls -l genomes/

Il faut donc retirer un du résultat précédent. Il y a donc **47 fichiers** dans le répertoire `genomes`.

Pour afficher exactement un fichier par ligne, sans ligne supplémentaire, on peut utiliser l'option `-1` (le chiffre 1) de la commande `ls` :

In [ ]:
#IN [8]
ls -1 genomes/

ce qui donne bien :

In [ ]:
#IN [9]
ls -1 genomes/ | wc -l


3. Familiarisez-vous avec les fichiers GenBank contenu dans le répertoire `genomes`. Combien de lignes y-a-t-il dans chaque fichiers ?

La commande pour compter le nombre de lignes d'un fichier est la commande `wc` avec l'option `-l`. Par exemple :

In [ ]:
#IN [10]
wc -l genomes/NC_000907_head.gbk

Le fichier `genomes/NC_000907_head.gbk` contient 100 lignes. Il faut maintenant réaliser la même chose pour les 46 autres fichiers. On peut le faire manuellement mais c'est assez fastidueux.

Une première façon d'automatiser ce comptage est d'utiliser un boucle `for` :

In [ ]:
#IN [11]
for name in genomes/*.gbk
do
    wc -l ${name}
done

Une seconde façon est d'utiliser directement la commande `wc` avec tous les fichiers GenBank :

In [ ]:
#IN [12]
wc -l genomes/*.gbk

Tous les fichiers contiennent 100 lignes chacun. `wc` affiche même le total : 47 fichiers x 100 lignes = 4700 lignes.

4. Quelle ligne des fichiers GenBank permet de savoir si l'organisme concerné est un staphylocoque ?

Par exemple, le fichier *NC_002976_head.gbk* est un fichier d'un staphylocoque alors que *NC_002505_head.gbk* non.


En observant les différents fichiers GenBank, on se rend compte que le nom d'organisme des staphylocoques est de la forme :
```
  ORGANISM  Staphylococcus epidermidis RP62A
```
C'est en particulier le mot `Staphylococcus` qui est discriminant ici.

En réalité d'autres lignes pourraient servir à identifier les staphylocoques, comme :
```
DEFINITION  Staphylococcus epidermidis RP62A, complete genome.
```
ou
```
SOURCE      Staphylococcus epidermidis RP62A
```

## Extraction automatique des staphylocoques

Supprimez maintenant le répertoire `genomes` :

Suppression du répertoire `genomes` :

In [ ]:
#IN [13]
rm -rf genomes/

**Attention** à la commande `rm -rf` qui va supprimer sans demander confirmation le répertoire fourni en argument. Sous Unix, il n'y a pas de corbeille, ni possibilité de revenir en arrière.

Si le fichier `genomes.tgz` n'est plus présent dans votre répertoire, pensez à le télécharger à nouveau.

Écrivez le script bash `get_staphylo.sh` qui s'exécute dans le même répertoire que `genomes.tgz` et qui :

1. décompresse l'archive `genomes.tgz` ;
2. crée un répertoire `staphylo`, au même niveau que le répertoire `genomes` qui contient les fichiers GenBank ;
3. décide si le fichier GenBank contient un génome de staphylocoques ou pas ;
4. affiche les noms des fichiers GenBank de staphylocoques ;
5. copie ces fichiers dans le répertoire `staphylo` ;
6. archive et compresse le répertoire `staphylo` sous le nom `staphylo.tgz` ;
7. supprime les répertoires `genomes` et `staphylo`.

Le script `get_staphylo.sh` devrait ressembler au script ci-dessous :

```
#! /usr/bin/env bash

# Pierre Poulain 2021

# Define directories with data
dir_input="genomes"
dir_output="staphylo"

# Uncompress initial file
tar zxf genomes.gz

# Create empty directory for Staphylococcus genomes
mkdir -p ${dir_output}

# Loop on all genbank files
for name in ${dir_input}/*.gbk
do
    # Select Staphylococcus files only
    if grep "ORGANISM" ${name} | grep "Staphylococcus"
    then
        # Print filename with Staphylococcus
        echo "${name} is a Staphylococcus"
        # Copy Staphylococcus file into the staphylo directory
        cp ${name} ${dir_output}
    fi
done

# Create final archive
tar zcf staphylo.tgz ${dir_output}/

# Delete temporary data directories
rm -rf ${dir_input} ${dir_output}
```

Copiez le contenu ci-dessus dans le script `get_staphylo.sh` puis exécutez ce script :

In [ ]:
#IN [14]
bash get_staphylo.sh

Pour terminer, exécutez ce même script depuis un terminal pour vérifier que vous obtenez bien le même résultat.

## Aide Notebook

*To make nice HTML reports with markdown:*  
- [html visualization tool 1](https://dillinger.io/)  
- [html visualization tool 2](https://stackedit.io/app#)  
- [to draw nice tables](https://www.tablesgenerator.com/markdown_tables)  
- [Ultimate guide](https://medium.com/analytics-vidhya/the-ultimate-markdown-guide-for-jupyter-notebook-d5e5abf728fd)

*Further reading on JupyterLab notebooks:*  
- [Jupyter Lab documentation](https://jupyterlab.readthedocs.io/en/latest/user/notebook.html)

*Here we are using JupyterLab interface implemented as part of the [Plasmabio](https://plasmabio.org/) project led by Sandrine Caburet, Pierre Poulain and Claire Vandiedonck.* 
*Merci beaucoup à Pierre Poulin pour son aide et son investissement dans la préparation des notebooks et de la présentation.*

*Last version revised on 22/09/2025 by Marwa ZIDI*